# Prediction model of intersection safety
(Notebook used to form the data used in the machine learning analysis)

by: Danny Allan, James Hatch, Andy Lelaucher


---

#Motivation and Context:

Neural networks are used in many fields to predict a lot of different events. While data comes in all shapes in sizes, more and more location-based data is becomming accessible to the public through platforms such as Open Street Maps (OSM), ArcGIS, Google Maps, and more.

That being said, predictive models are also prone to accumulate biases from their training data. This has led to the creation of models that, while displaying high categorical accuracy, are prone to making biased predictions in response to data. Since models are now being heavily incorporated into highly consequential fields such as incarceration and security, analyzing models for bias has become just as important as analyzing model accuracy.

Each year, thousands of people are killed in fatal car accidents. While accident-predicting models have the potential to save a lot of lives, biases in these models have the potential to reveal even deeper socioeconomic patterns (such as road maintenance and management relative to regional government funding differences). Ultimately, identifying and addressing these patterns might allow us to tackle the more systemic issues that are connected to fatal car crashes.

---

#Model Overview:

I intend to create a multimodal model that identifies fatal crash hotspots in U.S. cities using static street map images (collected using google static street view and fed into a CNN) and weather features (fed into an ANN). This model will be trained and tested using street intersection data collected using OSM (open street maps) and fatal car crash data provided by the FARS dataset.

In order to examine the development of biases in the trained model, I plan on using this model to predict the most common accident hotspots in Chicago. While having a high number of fatal car crashes each year, Chicago is a city know for having significant wealth disparities in different areas of the city. Since factors such as wealth may influence crash-related variables such as road maintenance, I hope to examine the model's predictions for patterns involving the most accident-prone areas in the city.

_Note: I was not able to make it to a full Chicago-level analysis this semester due to google API cost issues, but I will be able to run these tests next semester. Additionally, experimentation revealed that weather data is incompatible with my current model setup. As a result, I plan to use street features in my ANN instead._

##Target City
- Chicago (testing city - high population density)

---

##Predictive data
Street Imagery (CNN)
- Static street view google API
- OSM

Additional data
- Census data (estimated population sizes)

---

##Ground truth data
- FARs (fatal car accidents data)


Notes for the data:
- All data must come from 2015-2019 (inclusive) as 2020-2021 was COVID years and 2022-onward has corrupted pathways on the FARS dataset.


In [ ]:
# imports
import tensorflow as tf
import keras
import keras_hub
import numpy as np
import pandas as pd
import matplotlib as plt
import duckdb
import os
import re
from plotnine import *
import sklearn
import geopandas as gpd
import kagglehub

## Cloning the Github Repository

In [ ]:
# clones the github repository for use in google colab notebooks
from google.colab import userdata
accessToken = userdata.get('IntersectionToken')

# accesses the github dataset by cloning the repository and downloading its contents
repo_url = f"https://{accessToken}@github.com/SleepDeprived3/Intersection_Safety_Prediction.git"
!git clone {repo_url}

Cloning into 'Intersection_Safety_Prediction'...
remote: Enumerating objects: 10336, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 10336 (delta 40), reused 26 (delta 26), pack-reused 10289 (from 2)
Receiving objects: 100% (10336/10336), 386.33 MiB | 18.96 MiB/s, done.
Resolving deltas: 100% (113/113), done.
Updating files: 100% (10012/10012), done.


##Step 1 - Preliminary Analysis

I will start by acquiring data from the FARS (fatality analysis reporting system) dataset, a U.S. dataset containing information about fatal injuries from car crashes.

In [ ]:
# Chosen since street map images seem to be low in quality prior to 2015
min_year_of_data = 2015

# Chosen since some of the FARS data is corrupted between 2021-2023. Also, 2020-2021 was COVID-era
# which might skew results
max_year_of_data = 2018

list_of_years = range(min_year_of_data,max_year_of_data + 1)

In [ ]:
# Used to retrieve all data from the FARS dataset
for year in list_of_years:
  print(year)
  !wget https://static.nhtsa.gov/nhtsa/downloads/FARS/{year}/National/FARS{year}NationalCSV.zip
  !unzip FARS{year}NationalCSV.zip -d fars{year}

# FARS dataset guide: https://crashstats.nhtsa.dot.gov/Api/Public/ViewPublication/813706

In [ ]:
FARS_df_list = []

for year in list_of_years:
  # creating the path
  if year == 2015:
    yearPath = "/content/fars2015/FARS2015NationalCSV/accident.csv" # extra folder and lowercase csv letters made this an edge case
  elif year == 2018:
    yearPath = "/content/fars2018/accident.csv" # lowercase csv letters made this an edge case
  else: yearPath = f"/content/fars{year}/accident.CSV"

  # checking if the path exists, prints the path if not, creates a dataframe if it does exist
  if not os.path.exists(yearPath):
        print(f"Missing {year} path")
  else:
    year_df = duckdb.query(f"""
          SELECT
              ST_CASE, -- id used to match collision data from different CSVs
              STATENAME, -- Name of the state where the collision happened
              COUNTY,    -- GSA geographical code of the county where the collision happened
              CITY,      -- GSA geographical code of the city where the collision happened
              TYP_INT,   -- the type of intersection encountered (number id)
              MONTH, -- month of the collision
              DAY,   -- day of the collision
              YEAR,  -- year of the collision
              HOUR,  -- hour of the collision
              LATITUDE,    -- latitude coordinate of the collision  (to the seventh decimal point)
              LONGITUD,    -- longitude coordinate of the collision  (to the seventh decimal point)
              WEATHER,  -- the type of atmospheric conditions during the crash (number id)
              LGT_COND  -- the type and level of light during the crash (number id)
          FROM read_csv_auto('{yearPath}')
      """).df()
    FARS_df_list.append(year_df)

combined_FARS_df = pd.concat(FARS_df_list)

In [ ]:
combined_FARS_df
new_FARS = combined_FARS_df.copy()
new_FARS.dropna()
new_FARS
combined_FARS_df

In [ ]:
# filters non-city IDs out of the FARS data
combined_FARS_with_citynames_df = combined_FARS_df[~combined_FARS_df['CITY'].isin([0, 9997, 9898, 9999])]

# creates a dataframe that shows the cities with the greatest total number of fatal accidents
most_common_accident_locations = combined_FARS_with_citynames_df.groupby(['CITY', 'STATENAME']).size().reset_index(name='Count').sort_values(by='Count', ascending=False)
most_common_accident_locations.head(15)

In [ ]:
# USED TO FIND CITY BY CODE AND LOCATION BY FOUND CITY LAT/LONG
#combined_FARS_df[combined_FARS_df['CITY'] == ____]

Examining the longitude and latitutde coordinates of the most common accident cities gets you the following cities (in order ranked by total number of car crashes):

1. Los Angeles, CA (1091 fatalities) - https://www.kaggle.com/datasets/cityofLA/los-angeles-traffic-collision-data/data
2. Houston, TX (864 fatalities)
3. Phoenix, AZ (848 fatalities)
4. New York, NY (847 fatalities) - https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/about_data
5. Dallas, TX (703 fatalities)
6. San Antonio, TX (615 fatalities)
7. Jacksonville, FL (523 fatalities)
8. Chicago, IL (488 fatalities) - (limited) https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if/about_data
9. Detroit, MI (422 fatalities) - https://data.detroitmi.gov/maps/d837b05bdd9643698be30dfedbab0272/about
10. Memphis, TN (415 fatalities)
11. Indianapolis, IN (369 fatalities)
12. Philadelphia, PA (367 fatalities) - https://opendataphilly.org/datasets/crashes/
13. Fort Worth, TX (360 fatalities)
14. San Diego, CA (345 fatalities) - https://data.sandiego.gov/datasets/police-collisions/
15. Kinston, NC (336 fatalities)




In [ ]:
# importing pep census predictions (https://www.census.gov/programs-surveys/popest/technical-documentation/research/evaluation-estimates/2020-evaluation-estimates/2010s-cities-and-towns-total.html)
!wget https://www2.census.gov/programs-surveys/popest/datasets/2010-2020/cities/SUB-EST2020_ALL.csv

In [ ]:
# forms a pandas dataframe using the excel spreadsheet
population_estimates = pd.read_csv('/content/SUB-EST2020_ALL.csv', encoding="latin1", dtype={"STATE": str,"PLACE": str})
population_estimates_copy = population_estimates.copy()

# create two lists of strings containing the values of the relevant columns
list_of_years_str = []
for year in list(list_of_years):
  list_of_years_str.append("POPESTIMATE" + str(year))
relevant_columns = ["NAME", "STNAME", "STATE", "PLACE"] + list_of_years_str

# drop all rows with NA values
relevant_pop_estimates = population_estimates_copy[relevant_columns].dropna()

# find the mean estimated population of the provided population columns
relevant_pop_estimates["GEOID"] = relevant_pop_estimates["STATE"].str.zfill(2) + relevant_pop_estimates["PLACE"].str.zfill(5)
relevant_pop_estimates["mean_population"] = relevant_pop_estimates[list_of_years_str].mean(axis=1)


high_fatality_cities = ["Los Angeles city", "Houston city", "Phoenix city", "New York city", "Dallas city", "San Antonio city", "Jacksonville city", "Chicago city", "Detroit city", "Memphis city"]
high_fatality_cities_states = ["California", "Texas", "Arizona", "New York", "Texas", "Texas", "Florida", "Illinois", "Michigan", "Tennessee"]


relevant_pop_estimates[relevant_pop_estimates['NAME'].isin(high_fatality_cities) & relevant_pop_estimates['STNAME'].isin(high_fatality_cities_states)]

In [ ]:
# Copying information from the most_common_accident_locations dataframe
high_fatality_cities = ["Los Angeles city", "Houston city", "Phoenix city", "New York city", "Dallas city", "San Antonio city", "Jacksonville city", "Chicago city", "Detroit city", "Memphis city"]
high_fatality_cities_states = ["California", "Texas", "Arizona", "New York", "Texas", "Texas", "Florida", "Illinois", "Michigan", "Tenessee"]
fatalities_count = [1091,864,848,847,703,615,523,488,422,415]

# Calculates relative fatal incident percentages using estimated population means
# collected across the target years (and displayed in relevant_pop_estimates)
LA_percent = 1091 / 3961842.75
Houston_percent = 864 / 2304383.25
Pheonix_percent = 848 / 1621212.25
NYC_percent = 847 / 8441605.00
Dallas_percent = 703 / 1327103.25
San_Antonio_percent = 615 / 1498933.50
Jacksonville_percent = 523 / 886305.00
Chicago_percent = 488 / 2712931.75
Detroit_percent = 422 / 676407.50
Memphis_percent = 415 / 652648.25

# Adds relative fatalities to the previous lists to form a dataframe
relative_fatalities = [LA_percent, Houston_percent, Pheonix_percent, NYC_percent, Dallas_percent, San_Antonio_percent, Jacksonville_percent, Chicago_percent, Detroit_percent, Memphis_percent]

relative_fatalities_df = {
    "city" : high_fatality_cities,
    "state" : high_fatality_cities_states,
    "fatalities" : fatalities_count,
    "relative fatalities" : relative_fatalities
}

# Sorts the dataframe by relative fatalities to see which of the top ten cities
# for total fatal accident counts have the highest values relative to their
# population
relative_fatalities_df = pd.DataFrame(relative_fatalities_df)
relative_fatalities_df.sort_values(by="relative fatalities", ascending=False)

In order to collect a diverse range of photos for our dataset, I plan to use intersections from three cities in the list of the top ten cities with the most (total) fatal accidents (shown above). In order to choose these cities, I decided to use the following reasoning:

1. ***Memphis*** - the city in the top ten list with the most fatal crashes relative to its average estimated population size during the FARS dataset years range
2. ***New York City*** - the city in the top ten list with the least fatal crashes relative to its average estimated population size during the FARS dataset years range
3. ***Los Angeles*** - the city in the top ten list with the most a good mixture of road geometries for validation testing

Finally, when applying this model, I plan to use ***Chicago*** as my testing data location due to high wealth differences in its different counties.

The following section will be used to collect street view image data from the training datasets in order to allow for the creation of a accident hotspot model.

---
## Step 2 - Finding Intersections and Crash Location

In [ ]:
# installing programs that use Open Street Maps to create map networks
%pip install osmnx
import osmnx as ox
import networkx as nx
# for me: https://osmnx.readthedocs.io/en/stable/user-reference.html
# for me: https://networkx.org/documentation/stable/tutorial.html

def findIntersections(city):
  # retrieves a node graph from the cities of interest and identifies nodes
  # that mark intersections. also attempts to filter for non-standard
  # road intersections such as highway entrances, service roads, etc.
  G = ox.graph_from_place(city, network_type="drive")
  G = ox.project_graph(G)
  # prevent multi-node intersections from being counted twice by adding 15 meters of
  # overlapping node tolerance
  G = ox.consolidate_intersections(
    G,
    tolerance=15,
    rebuild_graph=True,
    dead_ends=False
  )
  Gu = ox.convert.to_undirected(G)
  intersection_nodes = [n for n, d in Gu.degree() if d >= 3]

  # creates a list of dictionaries for each intersection node that contains the
  # intersection id, x, and y
  intersections = []
  for node in intersection_nodes:
    data = G.nodes[node]
    intersections.append({
      "id": node,
      "x": data["x"],
      "y": data["y"]
    })

  # Creates a geodataframe containing the x and y info as well as the coordinate
  # reference system so the x and y coordinates can later be decoded into latitude
  # and longitude
  geodf = gpd.GeoDataFrame(
    intersections,
    geometry=gpd.points_from_xy(
        [r["x"] for r in intersections],
        [r["y"] for r in intersections]
    ),
    crs=G.graph["crs"]
  )

  # Use the Geodataframe to find longitude and latitude
  geodf_long_lat = geodf.to_crs("EPSG:4326")
  geodf["longitude"] = geodf_long_lat.geometry.x
  geodf["latitude"]  = geodf_long_lat.geometry.y

  return pd.DataFrame(geodf.drop(columns="geometry"))

In [ ]:
# Forming a list of cities for OSM to look up intersections for. This list contains the three (intended)
# training cities (New York City, Memphis, and Los Angeles), and one testing city (Chicago)
cities_of_interest = [{"city": "New York City", "state": "New York", "country": "United States"},
                      {"city": "Memphis", "state": "Tennessee", "country": "United States"},
                      {"city": "Los Angeles", "state": "California", "country": "United States"},
                      {"city": "Chicago", "state": "Illinois", "country": "United States"}]

In [ ]:
# Loading each of the intersection geodataframes using my previously defined method
# Doing this in individual cells to allow each one to be saved one at a time.
NYC_intersections = findIntersections(cities_of_interest[0])
MEM_intersections = findIntersections(cities_of_interest[1])
LA_intersections = findIntersections(cities_of_interest[2])
CHI_intersections = findIntersections(cities_of_interest[3])

In [ ]:
# Cell that creates subdataframes for each of the target cities in the FARS dataframe
NYC_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 4170) & (combined_FARS_df['STATENAME'] == "New York")]
MEM_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 1620) & (combined_FARS_df['STATENAME'] == "Tennessee")]
LA_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 1980) & (combined_FARS_df['STATENAME'] == "California")]
CHI_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 1670) & (combined_FARS_df['STATENAME'] == "Illinois")]

In [ ]:
# checking the number of crashes and number of intersections in memphis
print("Total number of intersections in memphis = " + str(MEM_intersections.size))
print("Total number of fatal accidents in memphis across all tested years = " + str(MEM_FARS.size))



---

Attempting to add some non-FARS data here... it should be noted that this data is much more limtied... as a result, here are the new four cities we will be using for this section of testing
1. Los Angeles, CA
2. New York City, NY
3. Detroit, MI
4. Philadelphia, PA

---



In [ ]:
# Forming a list of cities for OSM to look up intersections for. This list contains the three (intended)
# training cities (New York City, Memphis, and Los Angeles), and one testing city (Chicago)
mixed_cities_of_interest = [{"city": "New York City", "state": "New York", "country": "United States"},
                      {"city": "Detroit", "state": "Michigan", "country": "United States"},
                      {"city": "Los Angeles", "state": "California", "country": "United States"},
                      {"city": "Philadelphia", "state": "Pennsylvania", "country": "United States"}]

In [ ]:
# Loading each of the intersection geodataframes using my previously defined method
# Doing this in individual cells to allow each one to be saved one at a time.
NYC_intersections = findIntersections(mixed_cities_of_interest[0])
DET_intersections = findIntersections(mixed_cities_of_interest[1])
LA_intersections = findIntersections(mixed_cities_of_interest[2])
PHI_intersections = findIntersections(mixed_cities_of_interest[3])

In [ ]:
# Cell that creates subdataframes for each of the target cities in the FARS dataframe
NYC_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 4170) & (combined_FARS_df['STATENAME'] == "New York")]
DET_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 1260) & (combined_FARS_df['STATENAME'] == "Michigan")]
LA_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 1980) & (combined_FARS_df['STATENAME'] == "California")]
PHI_FARS = combined_FARS_df[(combined_FARS_df['CITY'] == 6540) & (combined_FARS_df['STATENAME'] == "Pennsylvania")]

In [ ]:
# Adding Fatal markers to each of these FARS dataframes for future reference
# Also reducing down to just the important columns to make combining easier later
# on
NYC_FARS = NYC_FARS[['LATITUDE', 'LONGITUD']]
NYC_FARS['FATAL'] = True

DET_FARS = DET_FARS[['LATITUDE', 'LONGITUD']]
DET_FARS['FATAL'] = True

LA_FARS = LA_FARS[['LATITUDE', 'LONGITUD']]
LA_FARS['FATAL'] = True

PHI_FARS = PHI_FARS[['LATITUDE', 'LONGITUD']]
PHI_FARS['FATAL'] = True

In [ ]:
# loading in all the city-level crash data (to supplement intersection data)

# Los Angeles
LA_local = pd.read_csv('/content/LA_ALL.csv')

# New York City
NYC_local = pd.read_csv('/content/NYC_ALL.csv')

# Detroit
DET_2015 = pd.read_csv('/content/Det_2015.csv')
DET_2016 = pd.read_csv('/content/Det_2016.csv')
DET_2017 = pd.read_csv('/content/Det_2017.csv')
DET_2018 = pd.read_csv('/content/Det_2018.csv')

# Philadelphai
Phil_2007_2017 = pd.read_csv('/content/Phil_2015-2017.csv')
Phil_2018_onward = pd.read_csv('/content/Phil_2018.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/LA_ALL.csv'

LA Local

In [ ]:
# creating a year column and filtering any years that aren't 2015-2018 (inclusive)
LA_local['year'] = (LA_local['Date Occurred'].str[:4])
LA_local = LA_local[LA_local['year'].isin(['2015', '2016', '2017', '2018'])]

# finding longitude and latitude coordinates by filtering for decimals, ints, and negatives
# longitude and latitude is presented in the string in a consistent order so we just create
# a list of numerical instances and retrieve either the first or second value in the list

# Function used for extracting longitude and latitude from the strings
def extract_coords(row_str):
    matches = re.findall(r'-?\d+\.?\d*', str(row_str))
    return float(matches[0]), float(matches[1])

# applying the function to form longitude and latitude coords for each row
LA_local[['LONGITUD', 'LATITUDE']] = LA_local['Location'].apply(lambda x: pd.Series(extract_coords(x)))

# filtering dataframe for just longitude and latitude
LA_local = LA_local[['LATITUDE', 'LONGITUD']]

# adding the fatal column
LA_local['FATAL'] = False

In [ ]:
LA_local.head()

In [ ]:
LA_FARS.head()

NYC Local

In [ ]:
# creating a year column and filtering any years that aren't 2015-2018 (inclusive)
NYC_local['year'] = (NYC_local['CRASH DATE'].str[6:])
NYC_local = NYC_local[NYC_local['year'].isin(['2015', '2016', '2017', '2018'])]

# relabling longitude and latitude coordinates
NYC_local['LATITUDE'] = (NYC_local['LATITUDE']).astype(float)
NYC_local['LONGITUD'] = (NYC_local['LONGITUDE']).astype(float)
NYC_local = NYC_local[['LATITUDE', 'LONGITUD']]

# dropping all NA longitude/latitude coordinate points as they are impossible to match otherwise
NYC_local = NYC_local.dropna()

# adding the fatal column
NYC_local['FATAL'] = False

In [ ]:
NYC_local.head()

In [ ]:
NYC_FARS.head()

Detroit Local

In [ ]:
DET_local = pd.DataFrame({'LATITUDE': [], 'LONGITUD': []})

for df in [DET_2015, DET_2016, DET_2017, DET_2018]:

  # creating a year column and filtering any years that aren't 2015-2018 (inclusive)
  df['year'] = (df['Crash Date'].str[:4])
  df = df[df['year'].isin(['2015', '2016', '2017', '2018'])]

  # relabeling longitude and latitude coordinates
  df['LONGITUD'] = (df['Longitude']).astype(float)
  df['LATITUDE'] = (df['Latitude']).astype(float)
  df = df[['LATITUDE', 'LONGITUD']]

  # dropping all NA longitude/latitude coordinate points as they are impossible to match otherwise
  df = df.dropna()

  # concatenating it to the DET_local dataframe
  DET_local = pd.concat([DET_local, df], ignore_index=True)

# adding the fatal column
DET_local['FATAL'] = False

In [ ]:
DET_local.head()

In [ ]:
DET_FARS.head()

Philadelphia Local

In [ ]:
PHI_local = pd.DataFrame({'LATITUDE': [], 'LONGITUD': []})

# 2015 - 2017 dataframe first
Phil_2007_2017['year'] = (Phil_2007_2017['crash_year'])
Phil_2007_2017 = Phil_2007_2017[Phil_2007_2017['year'].isin([2015, 2016, 2017, 2018])]

# relabeling longitude and latitude coordinates
Phil_2007_2017['LONGITUD'] = (Phil_2007_2017['dec_long']).astype(float)
Phil_2007_2017['LATITUDE'] = (Phil_2007_2017['dec_lat']).astype(float)
Phil_2007_2017 = Phil_2007_2017[['LATITUDE', 'LONGITUD']]

# dropping all NA longitude/latitude coordinate points as they are impossible to match otherwise
Phil_2007_2017 = Phil_2007_2017.dropna()


# ---


# 2018
Phil_2018_onward['year'] = (Phil_2018_onward['crash_year'])
Phil_2018_onward = Phil_2018_onward[Phil_2018_onward['year'].isin([2015, 2016, 2017, 2018])]

# relabeling longitude and latitude coordinates
Phil_2018_onward['LONGITUD'] = (Phil_2018_onward['dec_long']).astype(float)
Phil_2018_onward['LATITUDE'] = (Phil_2018_onward['dec_lat']).astype(float)
Phil_2018_onward = Phil_2018_onward[['LATITUDE', 'LONGITUD']]

# dropping all NA longitude/latitude coordinate points as they are impossible to match otherwise
Phil_2018_onward = Phil_2018_onward.dropna()


# ---


# concatenating it to the DET_local dataframe
PHI_local = pd.concat([Phil_2007_2017, Phil_2018_onward], ignore_index=True)

# adding the fatal column
PHI_local['FATAL'] = False

In [ ]:
PHI_local.head()

In [ ]:
PHI_FARS.head()

In [ ]:
# combines FARS and local dataframes
ALL_CRASH_LA = pd.concat([LA_local, LA_FARS], ignore_index=True)
ALL_CRASH_NYC = pd.concat([NYC_local, NYC_FARS], ignore_index=True)
ALL_CRASH_DET = pd.concat([DET_local, DET_FARS], ignore_index=True)
ALL_CRASH_PHI = pd.concat([PHI_local, PHI_FARS], ignore_index=True)

##IMPORTANT NOTE:
The google static street view API costs money when scraping over 10,000 images. As you can see from the above information, retrieving images from every crash site and intersection in every city would be too much for the scope of this project. As a result, we will be creating a smaller model using only one city for the time being.

In [ ]:
from numpy._core.arrayprint import format_float_scientific
# imports tools for creating GeoDataFrames, which will help convert longitude and latitude
# coordinates into meters
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree


# A function that matches FARS intersections with the intersection df.
# Returns an edited list of the intersections which includes the number of crashes that
# occur at that intersection. Also takes in the coordinate reference system of the
# target city's time zone as input.
def match_intersections(intersection_df, FARS_subframe, crs_input):
  # Distance limit between coordinates in meters
  max_dist = 30

  FARS_subframe = FARS_subframe[
    (FARS_subframe['LATITUDE'].between(-90, 90)) &
    (FARS_subframe['LONGITUD'].between(-180, 180))
  ].copy()

  # Setting up the return dataframe with 0 values for the number of accidents
  return_intersections = intersection_df.copy()
  return_intersections['numberAccidents'] = 0
  return_intersections['containsFatal'] = False

  # Create geodataframes for the FARS and intersection datasets for intersection matching
  geoIntersection = gpd.GeoDataFrame(
    intersection_df,
    geometry=gpd.points_from_xy(
        intersection_df['x'],
        intersection_df['y']
    ),
    crs=crs_input
  )

  geoFARS = gpd.GeoDataFrame(
    FARS_subframe,
    geometry=gpd.points_from_xy(FARS_subframe['LONGITUD'], FARS_subframe['LATITUDE']),
    crs='EPSG:4326'
  )

  # Convert longitude and latitude to meters by converting the FARS geodataframes into Intersection coordinate reference system
  FARS_meters = geoFARS.to_crs(geoIntersection.crs)

  # Takes X and Y coordinates from the converted coordinate reference systems
  FARS_meters['x'] = FARS_meters.geometry.x
  FARS_meters['y'] = FARS_meters.geometry.y

  # Used to find nearest K-neighbors to the intersections
  intersectionTree = BallTree(np.column_stack([geoIntersection.geometry.x, geoIntersection.geometry.y]), metric="euclidean")

  # Queries the intersection tree to provide relative location information for the crash data
  distance, index = intersectionTree.query(np.column_stack([FARS_meters.x, FARS_meters.y]), k=1)

  # Finds the closest intersection to a crash
  valid = distance[:, 0] <= max_dist
  matched_intersections = index[valid, 0]

  # Adds the matched intersection to the subframe
  #for index in matched_intersections:
  #  return_intersections.iloc[index, return_intersections.columns.get_loc("numberAccidents")] = return_intersections.iloc[index, return_intersections.columns.get_loc("numberAccidents")] + 1

  # rewriting the loop to include fatality markers
  matched_FARS_rows = FARS_subframe[valid].copy()
  matched_FARS_rows["matched_index"] = matched_intersections

  # groups intersections by their matched (valid) index
  # counts number of crashes and marks whether a fatal crash has occurred
  for _, group in matched_FARS_rows.groupby("matched_index"):
    index = group["matched_index"].iloc[0]
    return_intersections.iloc[index, return_intersections.columns.get_loc("numberAccidents")] += len(group)
    if group["FATAL"].any():
      return_intersections.iloc[index, return_intersections.columns.get_loc("containsFatal")] = True

  return return_intersections

In [ ]:
import math

# Finds the ESPG value for a given city based on it's longitude value.
# Used to turn longitude and latitude coordinates into meter-based coordinates
# since the curvature of the earth prevents a fully-accurate coordinate system
# for the whole world (and because we have to calculate EVERYTHING)
def findESPGValue(FARS_subframe):
  longitude = FARS_subframe['LONGITUD'].iloc[0]
  EPSG = "EPSG:326" + str(math.floor((longitude + 180) / 6) + 1)
  return(EPSG)

Matching intersections for the FARS-alone data

In [ ]:
'''
# creates a dataframe for each FAR/intersection pair that contains counts of the
# number of fatal accidents in each intersection and the coordinates of the intersection

# mmatch_intersections(NYC_intersections, NYC_FARS, findESPGValue(NYC_FARS)).groupby(['numberFatalAccidents']).count()
xyNYC = match_intersections(NYC_intersections, NYC_FARS, findESPGValue(NYC_FARS))
xyMEM = match_intersections(MEM_intersections, MEM_FARS, findESPGValue(MEM_FARS))
xyLA = match_intersections(LA_intersections, LA_FARS, findESPGValue(LA_FARS))
xyCHI = match_intersections(CHI_intersections, CHI_FARS, findESPGValue(CHI_FARS))
'''

Matching intersections for the FARS + supplementary data

In [ ]:
xyNYC = match_intersections(NYC_intersections, ALL_CRASH_NYC, findESPGValue(NYC_FARS))
xyLA = match_intersections(LA_intersections, ALL_CRASH_LA, findESPGValue(LA_FARS))
xyDET = match_intersections(DET_intersections, ALL_CRASH_DET, findESPGValue(DET_FARS))
xyPHI = match_intersections(PHI_intersections, ALL_CRASH_PHI, findESPGValue(PHI_FARS))

## Step 3 - Collecting Static Street View Images and OSM Data

Update for version 4: While I initially intended to collect weather data to scaffold some of the image data learning, I realized that weather data would be meaningless unless I switched my learning model to a time-based prediction model (which would be difficult given the spare frequency of crashes and the long time spans over which I am collecting data; if I were to restructure my model in this way, I would need to approach the problem in an entirely different way). As a result, I have decided to switch my model data to include street data from OSM. After examining the possible data types to include, I have found several viable options for predicting intersection fatalities.

- The number of lanes / road width - https://publichealth.jhu.edu/2023/narrower-lanes-safer-streets
- Traffic speed / speed limit / road class - https://www.emcinsurance.com/losscontrol/insights-d/2020/08/speed-increases-risk
- Street feature presence (stop sign, traffic light, crosswalk, sidewalk, bike lane, railroad, approach angle, etc.) - Partially implied in a pedestrian density study here: https://www.sciencedirect.com/science/article/pii/S0001457524002276


Ultimately, several of these features are already included in the image data. While initially hoped to scaffold my image data with street feature data, I also want to ensure that my model is learning something from the streetview images without relying too heavily on additional data. Since I am hoping that my CNN can pick up on visual cues (i.e. the number of lanes, road width, stop signs, traffic lights, street walks, and streetways), I want the rest of my information to include new information that cannot be derived from streetview imagery. This leaves me with the following categories:
- ***Traffic speed / speed limit*** (and existing speed limit differences)
- ***Road class***
- ***Approach angle*** (potential confound: this variable could loosely relate to specific cities. For example, New York city has much straighter roads (in my opinion) than San Fransisco)

---
**Note: Future research might investigate the inclusion of weather by turning fatality prediction into a time-based problem (i.e. grouping crashes by time and general city instead of by intersection). Image data would be less effective or entirely unnecessary in this model unless collected for each intersection at each time of the data.**

In [ ]:
from pyproj import Transformer

# A function that returns a dataframe containing approach view
# information for a given intersection
def find_approach_views(G, intersections_df):
  return_views = []

  # Explicitly construct a GeoDataFrame of road segments, ensuring 'u' and 'v' columns are present
  edges_list = []
  for u, v, key, data in G.edges(data=True, keys=True):
      if 'geometry' in data:
          edges_list.append({
              'u': u, # start node
              'u_node_attributes': dict(G.nodes[u]),
              'v': v, # end node
              'v_node_attributes': dict(G.nodes[v]),
              'key': key,
              'geometry': data['geometry'],
              **{k: v for k, v in data.items() if k not in ['u', 'v', 'key', 'geometry']}
          })
  edges_gdf = gpd.GeoDataFrame(edges_list, crs=G.graph["crs"])

  # Sets new variables equal to variables in the intersections dataframe
  for _, row in intersections_df.iterrows():
    node = row["id"]
    lat  = row["latitude"]
    lon  = row["longitude"]
    label = row["numberAccidents"]
    isFatal = row['containsFatal']

    # identifies approach road segments using starting and ending nodes
    inc_edges = edges_gdf[(edges_gdf["u"] == node) | (edges_gdf["v"] == node)]

    num_roads = 0

    #unique_edges = set(G.neighbors(node))
    #num_roads = len(unique_edges)

    # identifies an angle of attack for each road segment edge
    for _, edge in inc_edges.iterrows():

      # Get projected coordinates of the far and near points
      if edge["u"] == node: # edge goes from u node to v node

        p_far_projection = edge.geometry.coords[1]
        p_near_projection = edge.geometry.coords[0]

        # Collecting node attributes using the near node
        osm_highway_attributes_in_node = edge.u_node_attributes.get('highway') or {}
        traffic_light = ("traffic_signals" in osm_highway_attributes_in_node)
        traffic_stop = ("stop" in osm_highway_attributes_in_node)
        crosswalk = ("crossing" in osm_highway_attributes_in_node)
        camera = ("speed_camera" in osm_highway_attributes_in_node)
        speed_display = ("speed_display" in osm_highway_attributes_in_node)

      else: # edge goes v node to u node

        p_far_projection = edge.geometry.coords[-2]
        p_near_projection = edge.geometry.coords[-1]

        # Collecting node attributes using the near node
        osm_highway_attributes_in_node = edge.v_node_attributes.get('highway') or {}
        traffic_light = ("traffic_signals" in osm_highway_attributes_in_node)
        traffic_stop = ("stop" in osm_highway_attributes_in_node)
        crosswalk = ("crossing" in osm_highway_attributes_in_node)
        camera = ("speed_camera" in osm_highway_attributes_in_node)
        speed_display = ("speed_display" in osm_highway_attributes_in_node)

      # Calculate angle from the far point to the near point (towards the intersection)
      dx = p_near_projection[0] - p_far_projection[0]
      dy = p_near_projection[1] - p_far_projection[1]
      angle = (math.degrees(math.atan2(dx, dy)) + 360) % 360

      # defines the camera position (latitude and longitude) by moving the camera 15 meters opposite the
      # desired approach angle
      radian = math.radians(angle)
      dx = -15 * math.sin(radian)
      dy = -15 * math.cos(radian)
      camera_x = p_near_projection[0] + dx
      camera_y = p_near_projection[1] + dy

      transformer = Transformer.from_crs(G.graph["crs"], "EPSG:4326", always_xy=True)
      camera_lon, camera_lat = transformer.transform(camera_x, camera_y)

      # if a road is multiple types, turns a list of road types into a single type of road.
      # Since the roads we are looking at should MOSTLY be only one type of road, this check
      # really only exists to make sure that the output of this value is not a list.
      # Additionally, changes the road type to either be major or minor so as to avoid getting
      # trapped by poor learning in incredibly small road categories
      all_road_types = edge.get("highway", "unknown") # <----------------------------------------------------------------------------+++
      road_type = ""

      if isinstance(all_road_types, list):
        road_type = all_road_types[0]
      elif isinstance(all_road_types, str):
        road_type = all_road_types

      major_roads = ["motorway", "trunk", "primary", "secondary"]
      is_major = int(road_type in major_roads)

      # finds the data contained in the OSM edge
      # tries to predict speed limit if no speed limit is found...
      # ***NOTE: An external dataset would theoretically be more accurate, but matching it may be challenging?
      # for now, we will just be predicting based on typical U.S. speed limit conventions
      # while limit defaults vary from state to state, we do see some general patterns
      # reference: https://en.wikipedia.org/wiki/Speed_limits_in_the_United_States
      roadway_speed_defaults = {
        'motorway': 65,
        'trunk': 65,
        'primary': 65,
        'secondary': 65,
        'tertiary': 25,
        'residential': 25,
        'living_street': 25,
        'service': 25,
        'unclassified': 25,
        'motorway_link': 25,
        'trunk_link': 25,
        'primary_link': 25,
        'secondary_link': 25,
        'tertiary_link': 25,
      }

      speed_lim = np.nan

      if isinstance((edge.get("maxspeed", np.nan)), float):
        speed_lim = edge.get("maxspeed", np.nan) # <----------------------------------------------------------------------------++

      if ((np.isnan(speed_lim)) and (road_type in roadway_speed_defaults)):
        speed_lim = roadway_speed_defaults.get(road_type) # <----------------------------------------------------------------------------+++


      lights = edge.get("lit", "")
      bike_lane = edge.get("bicycle_road", "")
      lane_count = edge.get("lanes", np.nan)


      # Appends the appropriate information to the return_views list
      return_views.append({
          "intersection_id": node,
          "int_lat": lat,
          "int_lon": lon,

          # information for street view imagery
          "cam_lat": camera_lat,
          "cam_lon": camera_lon,
          "cam_heading": angle, # <---------------------------------------------------------------------------------------------+++

          # information for ANN
          "speed_limit": speed_lim,
          "road_type": road_type,
          "is_major_road": is_major,

          "has_crosswalk": crosswalk,
          "has_camera": camera,
          "has_speed_display": speed_display,
          "has_traffic_light": traffic_light,
          "has_stop_sign": traffic_stop,
          "number_of_connecting_roads": num_roads,

          "is_lit": lights,
          "has_bike_lane": bike_lane,
          "num_lanes": lane_count,

          # number of accidents
          "number_of_accidents": label, # <---------------------------------------------------------------------------------------------+++
          "has_had_fatal": isFatal
      })

  return pd.DataFrame(return_views)

In [ ]:
def findG(city):
  # retrieves a node graph from the cities of interest and identifies nodes
  # that mark intersections. also attempts to filter for non-standard
  # road intersections such as highway entrances, service roads, etc.
  G = ox.graph_from_place(city, network_type="drive")
  G = ox.project_graph(G)
  # prevent multi-node intersections from being counted twice by adding 15 meters of
  # overlapping node tolerance
  G = ox.consolidate_intersections(
    G,
    tolerance=15,
    rebuild_graph=True,
    dead_ends=False
  )
  return G

FARS-alone

In [ ]:
'''
# Creates node graphs for each city of interest
NYC_G = findG(cities_of_interest[0])
MEM_G = findG(cities_of_interest[1])
LA_G = findG(cities_of_interest[2])
CHI_G = findG(cities_of_interest[3])
'''

In [ ]:
'''
# NYC FARS

# uses the node graphs and XY coordinates to assign crashes to intersections
NYC_approach = find_approach_views(NYC_G, xyNYC)

# turns lists into tuples for dropping duplicate rows
for col in NYC_approach.columns:
    NYC_approach[col] = NYC_approach[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

# drops duplicate rows to keep only novel approaches
NYC_dropped_duplicates = (NYC_approach.copy()).drop_duplicates()

# counts the number of connecting roads by novel view
for index, row in NYC_dropped_duplicates.iterrows():
  NYC_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(NYC_dropped_duplicates[NYC_dropped_duplicates['intersection_id'] == row['intersection_id']])
'''

In [ ]:
'''
# MEM FARS

MEM_approach = find_approach_views(MEM_G, xyMEM)

for col in MEM_approach.columns:
    MEM_approach[col] = MEM_approach[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

MEM_dropped_duplicates = (MEM_approach.copy()).drop_duplicates()

for index, row in MEM_dropped_duplicates.iterrows():
  MEM_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(MEM_dropped_duplicates[MEM_dropped_duplicates['intersection_id'] == row['intersection_id']])
'''

In [ ]:
'''
# LA FARS

LA_approach = find_approach_views(LA_G, xyLA)

for col in LA_approach.columns:
    LA_approach[col] = LA_approach[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

LA_dropped_duplicates = (LA_approach.copy()).drop_duplicates()

for index, row in LA_dropped_duplicates.iterrows():
  LA_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(LA_dropped_duplicates[LA_dropped_duplicates['intersection_id'] == row['intersection_id']])
'''

In [ ]:
'''
# CHI FARS

CHI_approach = find_approach_views(CHI_G, xyCHI)

for col in CHI_approach.columns:
    CHI_approach[col] = CHI_approach[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

CHI_dropped_duplicates = (CHI_approach.copy()).drop_duplicates()

for index, row in CHI_dropped_duplicates.iterrows():
  CHI_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(CHI_dropped_duplicates[CHI_dropped_duplicates['intersection_id'] == row['intersection_id']])
'''

In [ ]:
'''
NYC_dropped_duplicates.to_csv('NYC_Crash_Locations_v3.csv', index=False)
MEM_dropped_duplicates.to_csv('MEM_Crash_Locations_v3.csv', index=False)
LA_dropped_duplicates.to_csv('LA_Crash_Locations_v3.csv', index=False)
CHI_dropped_duplicates.to_csv('CHI_Crash_Locations_v3.csv', index=False)
'''

In [ ]:
'''
NYC_dropped_duplicates['city'] = 'NYC'
MEM_dropped_duplicates['city'] = 'MEM'
LA_dropped_duplicates['city'] = 'LA'
CHI_dropped_duplicates['city'] = 'CHI'

#combinated_Crash = pd.concat([NYC_dropped_duplicates, MEM_dropped_duplicates,
                         #LA_dropped_duplicates, CHI_dropped_duplicates], ignore_index=True)
'''

In [ ]:
'''
NYC = pd.read_csv('/content/NYC_Crash_Locations.csv')
MEM = pd.read_csv('/content/MEM_Crash_Locations.csv')
LA = pd.read_csv('/content/LA_Crash_Locations.csv')
CHI = pd.read_csv('/content/CHI_Crash_Locations.csv')
'''

FARS + supplementary

In [ ]:
NYC_G = findG(mixed_cities_of_interest[0])
DET_G = findG(mixed_cities_of_interest[1])
LA_G = findG(mixed_cities_of_interest[2])
PHI_G = findG(mixed_cities_of_interest[3])

In [ ]:
# NYC MIXED

NYC_approach_mixed = find_approach_views(NYC_G, xyNYC)

for col in NYC_approach_mixed.columns:
    NYC_approach_mixed[col] = NYC_approach_mixed[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

NYC_dropped_duplicates = (NYC_approach_mixed.copy()).drop_duplicates()

for index, row in NYC_dropped_duplicates.iterrows():
  NYC_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(NYC_dropped_duplicates[NYC_dropped_duplicates['intersection_id'] == row['intersection_id']])

In [ ]:
# DET MIXED

DET_approach_mixed = find_approach_views(DET_G, xyDET)

for col in DET_approach_mixed.columns:
    DET_approach_mixed[col] = DET_approach_mixed[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

DET_dropped_duplicates = (DET_approach_mixed.copy()).drop_duplicates()

for index, row in DET_dropped_duplicates.iterrows():
  DET_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(DET_dropped_duplicates[DET_dropped_duplicates['intersection_id'] == row['intersection_id']])

In [ ]:
# LA MIXED

LA_approach_mixed = find_approach_views(LA_G, xyLA)

for col in LA_approach_mixed.columns:
    LA_approach_mixed[col] = LA_approach_mixed[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

LA_dropped_duplicates = (LA_approach_mixed.copy()).drop_duplicates()

for index, row in LA_dropped_duplicates.iterrows():
  LA_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(LA_dropped_duplicates[LA_dropped_duplicates['intersection_id'] == row['intersection_id']])

In [ ]:
# PHI MIXED

PHI_approach_mixed = find_approach_views(PHI_G, xyPHI)

for col in PHI_approach_mixed.columns:
    PHI_approach_mixed[col] = PHI_approach_mixed[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

PHI_dropped_duplicates = (PHI_approach_mixed.copy()).drop_duplicates()

for index, row in PHI_dropped_duplicates.iterrows():
  PHI_dropped_duplicates.at[index, 'number_of_connecting_roads'] = len(PHI_dropped_duplicates[PHI_dropped_duplicates['intersection_id'] == row['intersection_id']])

In [ ]:
NYC_dropped_duplicates.to_csv('Updated_NYC_CRASHES_2015-2018.csv', index=False)
DET_dropped_duplicates.to_csv('Updated_DET_CRASHES_2015-2018.csv', index=False)
LA_dropped_duplicates.to_csv('Updated_LA_CRASHES_2015-2018.csv', index=False)
PHI_dropped_duplicates.to_csv('Updated_PHI_CRASHES_2015-2018(2).csv', index=False)



---------
---
---
---
---
---

#This is the point in the code where the Github Data comes from

Notes
1. Does not currently have the API requests needed to actually draw images from google street view (that code exists below)
2. This code is not reduced in any way, and therefore might contain too much data and could go over the API limits. Methods for data reduction can be found below.
3. MOST IMPORTANTLY... a lot of the node-based boolean variables drawn from this data existed in OSM as either appearing or not exisisting at all. For example, a street light in OSM is either labelled as existing (True) or not labelled at all (could be either False or just no data is provided). Since OSM is crowd-sourced data, it is possible that this data could be innaccurate. More importantly though, the accuracy of this data is nearly impossible to verify without drawing data from a third source (maybe GIS or local, city-level data).

---

